# Gradient boosting (XGBoost / LightGBM) vs RandomForest

Requires optional packages from `requirements.txt`. If import fails, run `pip install xgboost lightgbm`.

In [ ]:
import sys
from pathlib import Path

_repo = Path.cwd().resolve()
for _ in range(12):
    if (_repo / "utils" / "data_paths.py").exists():
        sys.path.insert(0, str(_repo))
        break
    if _repo.parent == _repo:
        raise FileNotFoundError("Run from ml-notebook root.")
    _repo = _repo.parent

from utils.data_paths import data_dir
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv(data_dir() / "Bank_Personal_Loan_Modelling.csv")
X = df.drop(columns=["ID", "Personal Loan"])
y = df["Personal Loan"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print("RF test ROC-AUC:", roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]).round(4))

try:
    import xgboost as xgb
    xclf = xgb.XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, random_state=42,
        n_jobs=-1, eval_metric="logloss",
    )
    xclf.fit(X_train, y_train)
    print("XGB test ROC-AUC:", roc_auc_score(y_test, xclf.predict_proba(X_test)[:, 1]).round(4))
except ImportError as e:
    print("XGBoost not installed:", e)

try:
    import lightgbm as lgb
    lclf = lgb.LGBMClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=-1,
        subsample=0.9, colsample_bytree=0.9, random_state=42, n_jobs=-1,
    )
    lclf.fit(X_train, y_train)
    print("LGBM test ROC-AUC:", roc_auc_score(y_test, lclf.predict_proba(X_test)[:, 1]).round(4))
except ImportError as e:
    print("LightGBM not installed:", e)